# Week 3 Step 7: What should we do for this customer

The action is chosen by rules, not by the model. A high-risk, high-value customer gets a different action from a high-risk, low-value customer, and the same customer always gets the same action.

High value means lifetime value at or above the 75th percentile, which is calculated from the customer table. High risk is a churn probability of 0.61 or more. Low risk is 0.31 or less. Everyone else is medium risk.

Two modifiers can change the action after the base rule. An open ticket or a satisfaction score of 2 or below means resolve the issue before any offer. A failed payment in the 90 days before the feature cutoff turns the discount campaign into a payment-update reminder.

The note below each decision is the only part the model writes. The rule name is printed by the code.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.llm.retention import decide_action, example_rule_customers

examples = example_rule_customers()
for rule_id, customer_id in examples.items():
    decision = decide_action(customer_id)
    print(rule_id, decision["customer_id"], decision["status"])
    print(decision["rule_fired"])
    print("final action:", decision["action"], "| priority:", decision["priority"])
    print()


high_risk_high_value CUST000108 ok
High risk and high value -> Immediate personalized outreach
final action: Immediate personalized outreach | priority: High

high_risk_low_value CUST000002 ok
High risk and low value -> Automated promotional campaign
final action: Automated promotional campaign | priority: High

medium_risk_high_value CUST000003 ok
Medium risk and high value -> Proactive engagement | modifier: unresolved or low-satisfaction ticket
final action: Resolve the support issue before any offer | priority: Urgent

low_risk_high_value CUST000001 ok
Low risk and high value -> Reward and retain | modifier: unresolved or low-satisfaction ticket
final action: Resolve the support issue before any offer | priority: Urgent

standard_nurture CUST000004 ok
Medium or low risk and low value -> Standard nurture emails
final action: Standard nurture emails | priority: Low



The next cell asks the model to write the manager note for each of those five customers. It makes five model calls. `numbers_outside_facts` lists any digit in the note that is not already in the facts.


In [2]:
from src.llm.explain_churn import numbers_outside_facts
from src.llm.retention import recommend_action

for rule_id, customer_id in examples.items():
    result = recommend_action(customer_id)
    print(rule_id, result["status"])
    print(result["rule_fired"])
    print(result["message"])
    print("numbers not in the facts:", numbers_outside_facts(result["message"], result["facts"]))
    print()


high_risk_high_value ok
High risk and high value -> Immediate personalized outreach
Retention Note for CUST000108:

The rule "High risk and high value" fired for CUST000108. This is due to a High risk band and a High value tier. The final action is Immediate personalized outreach.
numbers not in the facts: []

high_risk_low_value ok
High risk and low value -> Automated promotional campaign
Retention Manager,

For CUST000002, the "High risk and low value" rule fired. This is due to a High risk band and Low value tier. The final action is an Automated promotional campaign.
numbers not in the facts: []

medium_risk_high_value ok
Medium risk and high value -> Proactive engagement | modifier: unresolved or low-satisfaction ticket
Retention Note for CUST000003

The rule "Medium risk and high value" fired for CUST000003. This is due to a Medium risk band and High value tier. The "unresolved or low-satisfaction ticket" modifier was applied because there is an open support ticket. The final act